# 5-Fold Stratified Cross-Validation Setup

This notebook sets up the cross-validation framework for model training. Because machine failures are rare (~3.4% of the data), a standard train/test split risks an unlucky split with too few failure examples in the test set. Stratified K-Fold ensures each fold preserves the same class distribution as the full dataset, giving a reliable, low-variance estimate of model performance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

In [ ]:
df = pd.read_csv("../data/processed/fused_dataset.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
target = 'Machine failure'

exclude_cols = ['UDI', 'Product ID', 'Type', 'timestamp', target,
                'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols].fillna(0)
y = df[target]

print("Number of features:", len(feature_cols))
print("Target distribution:\n", y.value_counts(normalize=True))

## Why Stratified K-Fold

With only ~3.4% positive class, a random split could easily produce a fold with very few or zero failure examples, making Macro F1 unstable or undefined. `StratifiedKFold` guarantees each of the 5 folds has approximately the same failure rate as the full dataset.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Number of splits: {skf.get_n_splits()}")

In [ ]:
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    train_failure_rate = y.iloc[train_idx].mean() * 100
    test_failure_rate = y.iloc[test_idx].mean() * 100
    print(f"Fold {fold}: Train failure rate = {train_failure_rate:.2f}% | "
          f"Test failure rate = {test_failure_rate:.2f}% | "
          f"Train size = {len(train_idx)} | Test size = {len(test_idx)}")

## Summary

The 5-fold stratified CV splitter is ready and confirmed to preserve the ~3.4% failure rate across all folds. This `skf` object will be reused in Issue #8 (SMOTE inside training folds) and Issue #9 (LightGBM training) to ensure consistent, leakage-free evaluation.